# Explore bronze Data

In [1]:
from processing.spark.utils.spark_session import create_spark_session

In [2]:
spark = create_spark_session("explore-bronze")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 13:54:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Movies

In [3]:
movies = spark.read.format("delta").load("s3a://lakehouse/bronze.db/batch/movies")

26/04/13 13:54:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
movies.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)



In [48]:
movies.limit(10).show(truncate = False)

+--------+----------------------------------+-------------------------------------------+------------------------+--------+
|movie_id|title                             |genres                                     |_ingestion_timestamp    |_source |
+--------+----------------------------------+-------------------------------------------+------------------------+--------+
|1       |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|2026-03-31 14:40:37.8339|postgres|
|2       |Jumanji (1995)                    |Adventure|Children|Fantasy                 |2026-03-31 14:40:37.8339|postgres|
|3       |Grumpier Old Men (1995)           |Comedy|Romance                             |2026-03-31 14:40:37.8339|postgres|
|4       |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |2026-03-31 14:40:37.8339|postgres|
|5       |Father of the Bride Part II (1995)|Comedy                                     |2026-03-31 14:40:37.8339|postgres|
|6      

In [49]:
movies.describe().show()

+-------+------------------+---------------+------------------+--------+
|summary|          movie_id|          title|            genres| _source|
+-------+------------------+---------------+------------------+--------+
|  count|            105071|         105071|            104629|  105071|
|   mean|168749.16193811802|           40.0|              null|    null|
| stddev| 81494.36957094916|           null|              null|    null|
|    min|                 1|         (1993)|(no genres listed)|postgres|
|    max|            300373|줄탁동시 (2012)|           Western|postgres|
+-------+------------------+---------------+------------------+--------+



26/03/31 14:48:05 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-13956540-4e8e-4743-940f-1dbb3765c047. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-13956540-4e8e-4743-940f-1dbb3765c047
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:177)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:113)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1231)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:62)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach(Array

#### Documentação - Pipeline Silver (movies)

**Colunas**

movieId

- Identificador único do filme.

title

- Título do filme.

genres

- Gêneros do filme. Quando há mais de um, eles aparecem separados pelo caractere “|”.


**Separar genres em tabela**

Tabela principal (silver.cleaned.movies)

```
movie_id
title
release_year
ingestion_timestamp
source
```

Tabela de relacionamento (silver.cleaned.movie_genres)
```
movie_id
genre
```

**Padronização de colunas**
```
movie_id -> movie_id
title -> title
genres -> genres_raw
```

**Separar título e ano**


```
Toy Story (1995)

title = Toy Story
release_year = 1995
```


**Tratar valores nulos**

```title NULL``` -> descartar (dado inválido)

```genres NULL``` -> preencher como ```unknown```

**Tipagem correta**

silver/cleaned/movies

<table align="left">
  <tr>
    <th>coluna</th>
    <th>tipo</th>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>title</td>
    <td>string</td>
  </tr>
  <tr>
    <td>release_year</td>
    <td>int</td>
  </tr>
  <tr>
    <td>genre</td>
    <td>string</td>
  </tr>
</table>

silver/enriched/movie_genres

<table align="left">
  <tr>
    <th>coluna</th>
    <th>tipo</th>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>genre</td>
    <td>string</td>
  </tr>

**Particionamento**

release_year

****

## Belief Data

In [12]:
belief = spark.read.parquet("s3a://lakehouse/bronze.db/batch/belief_data")

In [13]:
belief.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- is_seen: short (nullable = true)
 |-- watch_date: string (nullable = true)
 |-- user_elicit_rating: double (nullable = true)
 |-- user_predict_rating: double (nullable = true)
 |-- user_certainty: double (nullable = true)
 |-- tstamp: timestamp (nullable = true)
 |-- movie_idx: integer (nullable = true)
 |-- source: integer (nullable = true)
 |-- system_predict_rating: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)



In [14]:
belief.limit(10).show(truncate = False)

+-------+--------+-------+----------+------------------+-------------------+--------------+-------------------+---------+------+---------------------+--------------------------+--------+
|user_id|movie_id|is_seen|watch_date|user_elicit_rating|user_predict_rating|user_certainty|tstamp             |movie_idx|source|system_predict_rating|_ingestion_timestamp      |_source |
+-------+--------+-------+----------+------------------+-------------------+--------------+-------------------+---------+------+---------------------+--------------------------+--------+
|53982  |1       |-1     |          |-1.0              |-1.0               |-1.0          |2023-05-01 18:59:04|2        |2     |4.44785083170358     |2026-03-31 14:40:52.370411|postgres|
|56737  |1       |-1     |          |-1.0              |-1.0               |-1.0          |2023-10-08 13:52:36|7        |1     |3.57615421715505     |2026-03-31 14:40:52.370411|postgres|
|57704  |1       |-1     |          |-1.0              |-1.0     

In [15]:
belief.describe().show()

+-------+-----------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+---------------------+--------+
|summary|          user_id|          movie_id|            is_seen|         watch_date| user_elicit_rating|user_predict_rating|     user_certainty|         movie_idx|            source|system_predict_rating| _source|
+-------+-----------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+---------------------+--------+
|  count|          3004084|           3004084|            3004084|            3004084|            3004084|            3004084|            3004084|           3004084|           3004084|              3004084| 3004084|
|   mean|339709.6425702477|118793.36241563152|-0.9848233271772694|               null|-0.9765136394321863|-0.9614321703387788|-0.9585940

#### Documentação - Pipeline Silver (belief)

**Colunas**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>userId</td>
    <td>Identificador do usuário.</td>
  </tr>
  <tr>
    <td>movieId</td>
    <td>Identificador do filme.</td>
  </tr>
  <tr>
    <td>isSeen</td>
    <td>Indica se o usuário já viu o filme. (-1 = não respondeu, 0 = não viu, 1 = já viu)</td>
  </tr>
  <tr>
    <td>watchDate</td>
    <td>Data aproximada em que o usuário assistiu ao filme. Só possui valor quando isSeen = 1.</td>
  </tr>
  <tr>
    <td>userElicitRating</td>
    <td>Nota real informada pelo usuário para o filme quando ele já assistiu (escala de 0.5 a 5.0).</td>
  </tr>
  <tr>
    <td>userPredictRating</td>
    <td>Nota que o usuário acredita que daria ao filme caso ainda não tenha assistido (escala de 0.5 a 5.0).</td>
  </tr>
  <tr>
    <td>userCertainty</td>
    <td>Nível de confiança do usuário na previsão da nota (escala de 1 a 5).</td>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>Timestamp do momento em que a informação foi registrada.</td>
  </tr>
  <tr>
    <td>month_idx</td>
    <td>Índice que representa o mês do experimento em que o filme foi incluído no processo de elicitação.</td>
  </tr>
  <tr>
    <td>source</td>
    <td>Indica de qual grupo do processo de seleção o filme foi escolhido.</td>
  </tr>
  <tr>
    <td>systemPredictRating</td>
    <td>Nota prevista pelo sistema de recomendação para aquele usuário e filme.</td>
  </tr>
</table>

**Padronização de colunas**

Renomeação aplicada na camada CLEANED:

<table align="left">
  <tr>
    <th>original</th>
    <th>novo</th>
  </tr>
  <tr>
    <td>user_elicit_rating</td>
    <td>user_rating</td>
  </tr>
  <tr>
    <td>user_predict_rating</td>
    <td>predicted_rating</td>
  </tr>
  <tr>
    <td>system_predict_rating</td>
    <td>system_rating</td>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>event_timestamp</td>
  </tr>
  <tr>
    <td>_ingestion_timestamp</td>
    <td>ingestion_timestamp</td>
  </tr>
  <tr>
    <td>_source</td>
    <td>source_system</td>
  </tr>
</table>

**Tratamento de dados inválidos**

Valores sentinela (-1) são convertidos para NULL:

<table align="left"> 
    <tr> 
        <th>coluna</th> 
        <th>regra</th> 
    </tr> <tr> 
        <td>is_seen</td> 
        <td>-1 = NULL</td> 
    </tr> <tr> 
        <td>user_rating</td> 
        <td>-1 = NULL</td> 
    </tr> <tr> 
        <td>predicted_rating</td> 
        <td>-1 = NULL</td> 
    </tr> <tr> 
        <td>user_certainty</td> 
        <td>-1 = NULL</td> 
    </tr> </table>

**Tratamento de datas**

Regra: ```"" = NULL```

cast para `Date`

**Remoção de dados inválidos**

descartados:

- `user_id` é NULL
- `movie_id` é NULL

**Deduplicação**

Chave: `(user_id, movie_id, event_timestamp)`

Regra: Manter o mais recente via ``_ingestion_timestamp``

**Tipagem final (CLEANED)**
<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>user_id</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>is_seen</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>watch_date</td>
    <td>Tipo: date</td>
  </tr>
  <tr>
    <td>user_rating</td>
    <td>Tipo: double</td>
  </tr>
  <tr>
    <td>predicted_rating</td>
    <td>Tipo: double</td>
  </tr>
  <tr>
    <td>system_rating</td>
    <td>Tipo: double</td>
  </tr>
  <tr>
    <td>user_certainty</td>
    <td>Tipo: double</td>
  </tr>
  <tr>
    <td>event_timestamp</td>
    <td>Tipo: timestamp</td>
  </tr>
  <tr>
    <td>movie_idx</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>source_type</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>ingestion_timestamp</td>
    <td>Tipo: timestamp</td>
  </tr>
  <tr>
    <td>source_system</td>
    <td>Tipo: string</td>
  </tr>
  <tr>
    <td>processed_timestamp</td>
    <td>Tipo: timestamp</td>
  </tr>
</table>

**Features Criadas**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>rating_diff</td>
    <td>Diferença entre nota do usuário e do sistema.</td>
  </tr>
  <tr>
    <td>is_high_confidence</td>
    <td>Indica se user_certainty ≥ 4.</td>
  </tr>
  <tr>
    <td>has_watched</td>
    <td>Indica se is_seen == 1.</td>
  </tr>
  <tr>
    <td>event_date</td>
    <td>Data derivada do event_timestamp.</td>
  </tr>
</table>

**Regras de Negócio**


Diferença de Avaliação

- ``rating_diff = user_rating - system_rating``

(Somente quando ambos não são nulos)

Alta Confiança

- ``is_high_confidence = user_certainty >= 4``

Usuário Assistiu

- ``has_watched = is_seen == 1``

----

## Movie Elicitation

In [8]:
elicitation = spark.read.parquet("s3a://lakehouse/bronze.db/batch/movie_elicitation_set")

In [9]:
elicitation.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- month_idx: integer (nullable = true)
 |-- source: integer (nullable = true)
 |-- tstamp: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)



In [10]:
elicitation.limit(10).show(truncate = False)

+--------+---------+------+-------------------+--------------------------+--------+
|movie_id|month_idx|source|tstamp             |_ingestion_timestamp      |_source |
+--------+---------+------+-------------------+--------------------------+--------+
|1       |0        |1     |2023-02-27 19:30:03|2026-03-31 14:40:37.407635|postgres|
|1       |1        |1     |2023-04-01 00:01:47|2026-03-31 14:40:37.407635|postgres|
|1       |2        |1     |2023-05-01 00:02:01|2026-03-31 14:40:37.407635|postgres|
|1       |3        |1     |2023-06-01 00:01:40|2026-03-31 14:40:37.407635|postgres|
|1       |4        |1     |2023-07-01 00:01:56|2026-03-31 14:40:37.407635|postgres|
|1       |5        |1     |2023-08-01 00:01:40|2026-03-31 14:40:37.407635|postgres|
|2       |0        |1     |2023-02-27 19:30:03|2026-03-31 14:40:37.407635|postgres|
|2       |1        |1     |2023-04-01 00:01:47|2026-03-31 14:40:37.407635|postgres|
|2       |2        |1     |2023-05-01 00:02:01|2026-03-31 14:40:37.407635|po

In [11]:
elicitation.describe().show()

26/04/01 12:34:27 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+------------------+--------+
|summary|          movie_id|        month_idx|            source| _source|
+-------+------------------+-----------------+------------------+--------+
|  count|             84058|            84058|             84058|   84058|
|   mean|124368.98924552095|2.509945513811892|1.8462609150824432|    null|
| stddev| 111243.5848484193|3.001782359029911|1.1935393107451295|    null|
|    min|                 1|                0|                 1|postgres|
|    max|            299713|               14|                 5|postgres|
+-------+------------------+-----------------+------------------+--------+



#### Documentação - Pipeline Silver (movie_elicitation_set)

**Colunas**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>Identificador do filme.</td>
  </tr>
  <tr>
    <td>month_idx</td>
    <td>Índice do mês do experimento em que o filme foi exibido.</td>
  </tr>
  <tr>
    <td>source</td>
    <td>Grupo de seleção do filme.</td>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>Timestamp de quando o filme foi apresentado.</td>
  </tr>
  <tr>
    <td>_ingestion_timestamp</td>
    <td>Timestamp de ingestão do dado.</td>
  </tr>
  <tr>
    <td>_source</td>
    <td>Sistema de origem.</td>
  </tr>
</table>

**Regras de Negócio**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>1</td>
    <td>Popularidade.</td>
  </tr>
  <tr>
    <td>2</td>
    <td>Bem avaliado.</td>
  </tr>
  <tr>
    <td>3</td>
    <td>Lançamento recente popular.</td>
  </tr>
  <tr>
    <td>4</td>
    <td>Tendência.</td>
  </tr>
  <tr>
    <td>5</td>
    <td>Serendipidade.</td>
  </tr>
</table>


**Padronização de Colunas**

Renomeação aplicada na camada CLEANED:

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>event_timestamp</td>
  </tr>
  <tr>
    <td>_ingestion_timestamp</td>
    <td>ingestion_timestamp</td>
  </tr>
  <tr>
    <td>_source</td>
    <td>source_system</td>
  </tr>
  <tr>
    <td>source</td>
    <td>source_type</td>
  </tr>
</table>

**Tratamento de Dados Inválidos**

Registros descartados quando:

- ``movie_id`` é NULL
- ``event_timestamp`` é NULL

Validação de domínio

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>source_type</td>
    <td>Deve estar entre 1 e 5.</td>
  </tr>
  <tr>
    <td>month_idx</td>
    <td>Deve ser maior ou igual a 0.</td>
  </tr>
</table>

Valores nulos

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>month_idx</td>
    <td>Pode ser NULL (não quebra pipeline).</td>
  </tr>
  <tr>
    <td>source_type</td>
    <td>Pode ser NULL (tratado como desconhecido).</td>
  </tr>
</table>

**Deduplicação**

Chave de unicidade: ``(movie_id, event_timestamp)``

Regra: Manter o registro mais recente com base em ingestion_timestamp

**Tipagem Final (CLEANED)**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>month_idx</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>source_type</td>
    <td>Tipo: int</td>
  </tr>
  <tr>
    <td>event_timestamp</td>
    <td>Tipo: timestamp</td>
  </tr>
  <tr>
    <td>ingestion_timestamp</td>
    <td>Tipo: timestamp</td>
  </tr>
  <tr>
    <td>source_system</td>
    <td>Tipo: string</td>
  </tr>
  <tr>
    <td>processed_timestamp</td>
    <td>Tipo: timestamp</td>
  </tr>
</table>

**Features Criadas**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>event_date</td>
    <td>Data derivada do event_timestamp.</td>
  </tr>
  <tr>
    <td>is_recent</td>
    <td>Indica se o filme foi exibido recentemente.</td>
  </tr>
  <tr>
    <td>source_category</td>
    <td>Descrição categórica do source.</td>
  </tr>
  <tr>
    <td>month_group</td>
    <td>Agrupamento de meses (ex: início, meio, fim do experimento).</td>
  </tr>
</table>

**Regras de Negócio**

Data do Evento: ``event_date = to_date(event_timestamp)``

Categoria do Source: 

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>1</td>
    <td>popular</td>
  </tr>
  <tr>
    <td>2</td>
    <td>well_rated</td>
  </tr>
  <tr>
    <td>3</td>
    <td>recent_popular</td>
  </tr>
  <tr>
    <td>4</td>
    <td>trending</td>
  </tr>
  <tr>
    <td>5</td>
    <td>serendipity</td>
  </tr>
</table>

Agrupamento de Mês (feature analítica)

Regra:

``month_idx`` entre 0–2 -> *early_stage*

``month_idx`` entre 3–6 -> *mid_stage*

``month_idx`` > 6 -> *late_stage*

Recência do Evento 

``is_recent`` = ``event_timestamp >= current_date - X dias``

--- 

## Ratings

In [4]:
ratings = spark.read.parquet("s3a://lakehouse/bronze.db/streaming/ratings_for_additional_users")

26/04/01 16:53:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
ratings.printSchema()

root
 |-- userId: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- tstamp: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)
 |-- _topic: string (nullable = true)



In [6]:
ratings.limit(10).show(truncate = False)

+------+-------+------+-------------------+--------------------------+-------+--------------------------------+
|userId|movieId|rating|tstamp             |_ingestion_timestamp      |_source|_topic                          |
+------+-------+------+-------------------+--------------------------+-------+--------------------------------+
|328221|187505 |5     |2018-12-05 23:10:11|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187541 |5     |2018-11-06 21:39:15|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187577 |5     |2022-02-22 14:20:19|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187593 |4     |2018-10-12 08:20:55|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187601 |5     |2018-12-05 23:33:37|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187717 |4     |2020-10-26 14:35:06|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional

In [7]:
ratings.describe().show()

26/04/01 16:53:35 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+------------------+-------------------+-------+--------------------+
|summary|            userId|          movieId|            rating|             tstamp|_source|              _topic|
+-------+------------------+-----------------+------------------+-------------------+-------+--------------------+
|  count|           6231812|          6231812|           6231812|            6231812|6231812|             6231812|
|   mean| 290728.1072395958|77419.10746810077|2.9164129659123357|               null|   null|                null|
| stddev|101041.49218202523|82183.53348236556|1.7366988163045725|               null|   null|                null|
|    min|            100109|                1|                -1|1997-09-15 16:09:20|  kafka|csv-ratings_for_a...|
|    max|             99682|            99999|                NA|2024-05-05 21:45:54|  kafka|csv-ratings_for_a...|
+-------+------------------+-----------------+------------------+---------------

#### Documentação - Pipeline Silver (ratings)

**Colunas**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descricao</th>
  </tr>
  <tr>
    <td>userId</td>
    <td>Identificador do usuário.</td>
  </tr>
  <tr>
    <td>movieId</td>
    <td>Identificador do filme avaliado.</td>
  </tr>
  <tr>
    <td>rating</td>
    <td>Nota dada pelo usuário ao filme. Escala de 0.5 a 5.0 estrelas.</td>
  </tr>
  <tr>
    <td>timestamp</td>
    <td>Momento em que a avaliação foi feita, em formato Unix timestamp (segundos desde 01/01/1970).</td>
  </tr>
</table>

**Padronização de Colunas**

<table align="left">
  <tr>
    <th>original</th>
    <th>novo</th>
  </tr>
  <tr>
    <td>userId</td>
    <td>user_id</td>
  </tr>
  <tr>
    <td>movieId</td>
    <td>movie_id</td>
  </tr>
  <tr>
    <td>rating</td>
    <td>rating</td>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>event_timestamp</td>
  </tr>
  <tr>
    <td>_ingestion_timestamp</td>
    <td>ingestion_timestamp</td>
  </tr>
  <tr>
    <td>_source</td>
    <td>source_system</td>
  </tr>
  <tr>
    <td>_topic</td>
    <td>source_topic</td>
  </tr>
</table>

**Regras de Negócio (Domínio)**

``rating`` entre ``0.5 AND 5.0`` se não ``NULL``

**Conversão de tipos**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>tipo final</th>
  </tr>
  <tr>
    <td>user_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>rating</td>
    <td>double</td>
  </tr>
  <tr>
    <td>event_timestamp</td>
    <td>timestamp</td>
  </tr>
</table>

**Tratamento de nulos obrigatórios**

Registros descartados se :

- ``user_id`` NULL

- ``movie_id`` NULL

- ``event_timestamp`` NULL

**Tratamento de rating inválido**

``NA`` - NULL  
``-1`` - NULL  
``fora do range``- NULL

**Deduplicação**

``(user_id, movie_id, event_timestamp)``

Regra: manter o mais recente ``ingestion_timestamp``

**Tipagem Final (CLEANED)**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>tipo</th>
  </tr>
  <tr>
    <td>user_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>rating</td>
    <td>double</td>
  </tr>
  <tr>
    <td>event_timestamp</td>
    <td>timestamp</td>
  </tr>
  <tr>
    <td>ingestion_timestamp</td>
    <td>timestamp</td>
  </tr>
  <tr>
    <td>source_system</td>
    <td>string</td>
  </tr>
  <tr>
    <td>source_topic</td>
    <td>string</td>
  </tr>
  <tr>
    <td>processed_timestamp</td>
    <td>timestamp</td>
  </tr>
</table>

Flag positiva

``is_positive`` = ``rating >= 4``

---

## Recommendation History

In [3]:
history = spark.read.parquet("s3a://lakehouse/bronze.db/streaming/user_recommendation_history")

26/04/02 12:07:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
history.printSchema()

root
 |-- userId: string (nullable = true)
 |-- tstamp: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- predictedRating: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)
 |-- _topic: string (nullable = true)



In [5]:
history.limit(10).show(truncate = False)

+------+----------+-------+----------------+--------------------------+-------+-------------------------------+
|userId|tstamp    |movieId|predictedRating |_ingestion_timestamp      |_source|_topic                         |
+------+----------+-------+----------------+--------------------------+-------+-------------------------------+
|377084|1677651051|296    |4.62638615403935|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|55820  |4.34364173570518|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|924    |4.30338526694785|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|1732   |4.2554163884785 |2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|1258   |4.24126920909493|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|1201   |4.21599746536257|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_h

In [7]:
history.describe().show(truncate = False)

+-------+------------------+--------------------+-----------------+------------------+-------+-------------------------------+
|summary|userId            |tstamp              |movieId          |predictedRating   |_source|_topic                         |
+-------+------------------+--------------------+-----------------+------------------+-------+-------------------------------+
|count  |1285664           |1285664             |1285664          |1285664           |1285664|1285664                        |
|mean   |342884.52245687833|1.696552517096722E9 |78507.92973280733|4.430003836294816 |null   |null                           |
|stddev |79874.55142619881 |1.0594145024528226E7|93791.91794814276|0.4177386976085752|null   |null                           |
|min    |101362            |1677651051          |1                |0.444623023621201 |kafka  |csv-user_recommendation_history|
|max    |93990             |1714521553          |99996            |5                 |kafka  |csv-user_recommen

#### Documentação - Pipeline Silver (user_recommendation_history)

**Colunas**

<table align="left">
  <tr>
    <th>coluna original</th>
    <th>coluna final</th>
    <th>descrição</th>
  </tr>
  <tr>
    <td>userId</td>
    <td>user_id</td>
    <td>Identificador do usuário</td>
  </tr>
  <tr>
    <td>movieId</td>
    <td>movie_id</td>
    <td>Identificador do filme</td>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>event_timestamp</td>
    <td>Timestamp da recomendação</td>
  </tr>
  <tr>
    <td>predictedRating</td>
    <td>predicted_rating</td>
    <td>Nota prevista pelo modelo</td>
  </tr>
  <tr>
    <td>_ingestion_timestamp</td>
    <td>ingestion_timestamp</td>
    <td>Momento de ingestão</td>
  </tr>
  <tr>
    <td>_source</td>
    <td>source_system</td>
    <td>Sistema de origem</td>
  </tr>
  <tr>
    <td>_topic</td>
    <td>source_topic</td>
    <td>Tópico Kafka</td>
  </tr>
</table>

**Tipagem Final**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>tipo</th>
  </tr>
  <tr>
    <td>user_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>predicted_rating</td>
    <td>double</td>
  </tr>
  <tr>
    <td>event_timestamp</td>
    <td>timestamp</td>
  </tr>
  <tr>
    <td>ingestion_timestamp</td>
    <td>timestamp</td>
  </tr>
  <tr>
    <td>source_system</td>
    <td>string</td>
  </tr>
  <tr>
    <td>source_topic</td>
    <td>string</td>
  </tr>
  <tr>
    <td>processed_timestamp</td>
    <td>timestamp</td>
  </tr>
</table>

**Tratamento de Dados Inválidos**

Tratamento de Dados Inválidos:

- ``user_id`` é NULL
- ``movie_id`` é NULL
- ``event_timestamp`` é NULL

**Validação de Domínio**

``predicted_rating`` entre [0.0 - 5.0] senão NULL

**Deduplicação**

``(user_id, movie_id, event_timestamp)``

Manter o mais recente com base em ``ingestion_timestamp``

**Features Criadas**

``event_date`` = ``to_date(event_timestamp)``

**Classificação do score previsto**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>descrição</th>
  </tr>
  <tr>
    <td>predicted_rating_category</td>
    <td>Classificação da nota prevista</td>
  </tr>
</table>

Regra: 

<table align="left">
  <tr>
    <th>faixa</th>
    <th>categoria</th>
  </tr>
  <tr>
    <td>&lt;= 2.5</td>
    <td>low</td>
  </tr>
  <tr>
    <td>&lt;= 3.5</td>
    <td>medium</td>
  </tr>
  <tr>
    <td>&gt; 3.5</td>
    <td>high</td>
  </tr>
</table>

**Recomendação forte**

``is_high_score`` - is_high_score

``predicted_rating`` >= ``4.0`` - true

**Bucketização do score**

``predicted_rating_bucket`` - predicted_rating_bucket

<table align="left">
  <tr>
    <th>aixa</th>
    <th>bucket</th>
  </tr>
  <tr>
    <td>0–1.5</td>
    <td>very_low</td>
  </tr>
  <tr>
    <td>1.5–2.5</td>
    <td>low</td>
  </tr>
  <tr>
    <td>2.5–3.5</td>
    <td>mid</td>
  </tr>
  <tr>
    <td>3.5–4.5</td>
    <td>high</td>
  </tr>
  <tr>
    <td>4.5–5.0</td>
    <td>top</td>
  </tr>
</table>

---

In [6]:
spark.stop()